In [ ]:
import pandas as pd

# Se carga el archivo para poder leer "manipular"
data_frame = pd.read_csv('server_logs.csv')
#print(data_frame.to_string())   ->  Print de verificacion

# Convertir el timestamp_event en objetos tipo datetime
data_frame['timestamp_event'] = pd.to_datetime(data_frame['timestamp_event'])
# print(data_frame['timestamp_event'])  -> Print de verificacion

# Creamos una columna para verificar "Bad Event"
data_frame['malo'] = (data_frame['severity'].isin(['ERROR', 'CRITICAL'])) | (data_frame["status_code"] >= 500)
#print(data_frame['malo'])   #->  Print de verificacion

#Definiciones Operativas - Time Window(bin) - Agrupacion en ventanas de 5 minutos
data_frame_resumen = data_frame.set_index('timestamp_event').resample('5min').agg(
    total_events = ('service_name', 'count'),
    bad_events = ('malo', 'sum'),
    #average_latency = ('latency_ms', 'mean')     # Averiguar bien en que etapa y para que se usa
)

# Calculo del Bad Rate
data_frame_resumen['badRate'] = data_frame_resumen['bad_events'] / data_frame_resumen['total_events']
# print(data_frame_resumen['Bad Rate'])   -> Print de verificacion

# Deteccion momento critico, total_events (minimo 20), ordenamo por el peor Bad Rate
peores_ventanas = data_frame_resumen[data_frame_resumen['total_events'] >= 20].sort_values('badRate', ascending=False)
inicio_momento_critico = peores_ventanas.index[0]

# Diagnostico de lo ocurrido en los 5min
# Se filtra el data_frame original, solo para ese momento
data_inicident =  data_frame[(data_frame['timestamp_event'] >= inicio_momento_critico) &
                            (data_frame['timestamp_event'] < (inicio_momento_critico + pd.Timedelta(minutes=5)))]

# Comparacion del incidente vs Baseline(baseline es todo lo que NO es el momento critico, incidente)
data_baseline = data_frame[data_frame['timestamp_event'] != inicio_momento_critico] 


Total de logs

In [28]:

total_logs = data_frame.shape[0]
total_logs

5795

Por severidad

In [29]:
#data_frame['severity'].unique()
#data_frame.groupby('severity').size()

tiposDeSeveridad = data_frame['severity'].value_counts()
severidadMasComun = data_frame['severity'].value_counts().idxmax()
severidadMasComun


'INFO'

Servicio con mas logs

In [30]:
# Conteo de servicios y cantidad de apariciones
conteoLogs = data_frame['service_name'].value_counts()
servicioMasLogs = data_frame['service_name'].value_counts().idxmax()
servicioMasLogs

'api-gateway'

Servicio con menos logs

In [31]:
servicioMenosLogs = data_frame['service_name'].value_counts().idxmin()  
servicioMenosLogs

'notification-service'

Mensaje mas frecuente

In [32]:
mensajeMasRepetido = data_frame['message'].value_counts().idxmax()
mensajeMasRepetido

'Health check OK'

Mensaje "malo" mas frecuente

In [33]:
mensaje_malo_mas_repetido = data_frame[data_frame['malo'] == True].value_counts('message').idxmax()
mensaje_malo_mas_repetido


'Order creation failed - inventory lock timeout'

Tabla de representacion de exploracion inicial

In [ ]:
# Creacion de dataFrame vacio
df = pd.DataFrame(columns=['Total de logs', 'Severidad mas comun', 'Servicio con mas logs', 'Servicio con menos logs', 'Mensaje mas repetido', 'Mensaje malo mas repetido'])
df.loc[0] = [total_logs, severidadMasComun, servicioMasLogs, servicioMenosLogs, mensajeMasRepetido, mensaje_malo_mas_repetido]
df

,Total de logs,Severidad mas comun,Servicio con mas logs,Servicio con menos logs,Mensaje mas repetido,Mensaje malo mas repetido
0,5795,INFO,api-gateway,notification-service,Health check OK,Order creation failed - inventory lock timeout


Tabla de deteccion del momento critico (tabla top 5)

In [35]:
top5_peoresMomentos = data_frame_resumen[data_frame_resumen['total_events'] >= 20].sort_values('badRate', ascending=False).head(5)
top5_peoresMomentos


,total_events,bad_events,average_latency,badRate
timestamp_event,,,,
2026-01-10 11:10:00+00:00,189,110,1589.687831,0.582011
2026-01-10 11:15:00+00:00,228,129,1572.394737,0.565789
2026-01-10 11:20:00+00:00,111,59,1429.801802,0.531532
2026-01-11 14:35:00+00:00,255,117,1526.662745,0.458824
2026-01-11 14:30:00+00:00,156,68,1395.570513,0.435897
